In [1]:
import glob
import os 
import copy
import sys
import itertools
import random
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import models
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger

In [5]:
import sklearn
print ("numpy: ", np.__version__)
print ("pandas: ", pd.__version__)
print ("sklearn: ", sklearn.__version__)

numpy:  2.4.3
pandas:  3.0.2
sklearn:  1.8.0


In [6]:
path="./"

In [ ]:
seed = 0  # 0-9


In [10]:
model="CL3"

In [11]:
latent_list=[2,4,8,16,32,64,128,256,512]

In [ ]:
proj_dim=16

# 1. Loading Data

In [7]:
df_param=pd.read_csv(path+'/data/params/param_sum.csv', index_col=0)
df_param

,DSC_Tg,"DSC_deltaH,cryst","DSC_deltaH,melt",IR1_mu1,IR1_sigma1,IR1_Int1,IR1_mu2,IR1_sigma2,IR1_Int2,IR2_mu1,...,WAXS_Int10,WAXS_mu11,WAXS_sigma11,WAXS_Int11,Process_Temp,Process_Time,Process_NucConc,Property_Deg,Property_StrainBreak,Property_Young
17_1,60.466362,-47.596012,46.480912,925.000000,8.000000,2.763751e-01,955.131085,6.280011,7.863692,640.601782,...,2660.815097,22.709726,1.452573,106.494511,75,5,0.000,0.078653,0.159800,1533.741152
18_1,61.271702,-48.692377,49.326462,925.000000,8.000000,1.026984e-02,955.259577,5.986762,7.722477,640.861463,...,2290.375137,22.817104,1.521605,94.686515,75,5,0.167,0.086981,0.088500,1595.907404
19_1,62.323349,-15.013878,51.449458,924.157782,4.768348,2.492578e+00,955.615411,5.403198,5.721598,641.002109,...,1377.228359,24.895944,5.824055,1981.239121,75,5,0.500,0.081722,0.057200,1704.734993
21_1,60.937294,-46.231835,44.462586,924.999997,0.000508,1.373156e-10,955.284399,5.796023,7.480258,640.820021,...,2342.947158,24.613239,6.064391,2356.525132,75,10,0.000,0.078401,0.079833,1549.705975
22_1,61.153538,-43.927379,50.978043,925.000000,8.000000,3.040490e-12,955.141682,6.047396,7.448746,640.502364,...,1933.092536,24.754999,5.892152,2031.016989,75,10,0.167,0.087794,0.177067,1499.181431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78_3,59.287018,-1.101862,45.724219,922.401149,5.598432,6.180660e+00,956.429912,5.037394,5.100007,640.000000,...,1042.660663,23.871538,4.654210,1827.687507,120,40,0.167,0.009879,0.034867,1853.110523
79_3,57.929401,-4.225821,42.851180,922.909254,5.136471,5.767580e+00,956.330586,5.145190,4.492747,642.000000,...,612.843045,22.823962,1.414205,133.393312,120,40,0.500,0.010791,0.039867,1812.193412
81_3,61.119339,-41.676676,47.124293,925.000000,8.000000,2.036504e-11,955.205546,6.240761,8.285110,640.686571,...,1978.966766,25.000000,5.593697,2060.534350,0,0,0.000,0.080662,0.116667,1587.940605
82_3,60.671047,-38.773534,48.770991,924.876810,7.999546,8.625324e-11,955.241218,5.947278,7.917637,640.750933,...,2269.634558,24.983987,5.620694,2226.831391,0,0,0.167,0.078363,0.099367,1442.482979


In [8]:
df_dc=df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9']].sum(axis=1)/df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9','WAXS_Int10','WAXS_Int11']].sum(axis=1)
df_dc

17_1    0.042973
18_1    0.049297
19_1    0.128264
21_1    0.022538
22_1    0.024244
          ...   
78_3    0.261020
79_3    0.595242
81_3    0.025614
82_3    0.024128
83_3    0.024049
Length: 153, dtype: float64

In [9]:
df_param=pd.read_csv(path+'/data/params/param_sum.csv',index_col=0)
df_pred=pd.concat([df_param[['Property_Deg', 'Property_StrainBreak', 'Property_Young']], df_dc], axis=1)
df_pred['Property_Deg']=df_pred['Property_Deg']*100
df_pred.columns=["Deg","Strain","Young","CrysDeg"]
df_pred["CrysDeg"]=df_pred["CrysDeg"]*100
df_pred

,Deg,Strain,Young,CrysDeg
17_1,7.8653,0.159800,1533.741152,4.297316
18_1,8.6981,0.088500,1595.907404,4.929744
19_1,8.1722,0.057200,1704.734993,12.826432
21_1,7.8401,0.079833,1549.705975,2.253836
22_1,8.7794,0.177067,1499.181431,2.424432
...,...,...,...,...
78_3,0.9879,0.034867,1853.110523,26.101993
79_3,1.0791,0.039867,1812.193412,59.524230
81_3,8.0662,0.116667,1587.940605,2.561380
82_3,7.8363,0.099367,1442.482979,2.412784


In [10]:
df_pred=df_pred.round({
    'Deg': 3,  
    'Strain': 4,  
    'Young': 0,  
    'CrysDeg': 2,  
})
df_pred

,Deg,Strain,Young,CrysDeg
17_1,7.865,0.1598,1534.0,4.30
18_1,8.698,0.0885,1596.0,4.93
19_1,8.172,0.0572,1705.0,12.83
21_1,7.840,0.0798,1550.0,2.25
22_1,8.779,0.1771,1499.0,2.42
...,...,...,...,...
78_3,0.988,0.0349,1853.0,26.10
79_3,1.079,0.0399,1812.0,59.52
81_3,8.066,0.1167,1588.0,2.56
82_3,7.836,0.0994,1442.0,2.41


In [11]:
df_IR=pd.read_csv(path+'/data/measurement/IR.csv', index_col=0)
df_WAXS=pd.read_csv(path+"/data/measurement/WAXS.csv", index_col=0)
POM_list=np.load(path+'/data/measurement/POM2.npy')

In [12]:
df_IR_norm=df_IR.div(df_IR.sum(axis=1), axis=0)
df_WAXS_norm=df_WAXS.div(df_WAXS.sum(axis=1), axis=0)
POM_list_norm=POM_list/np.max(POM_list)

In [185]:
class AllDataset(Dataset):
    def __init__(self, X_IR, X_WAXS, X_POM, y):
        # (N, length) → (N, 1, length)
        X_IR = X_IR[:, None, :]
        X_WAXS = X_WAXS[:, None, :]
        X_POM = X_POM[:, None, :]
        
        self.X1 = torch.tensor(X_IR, dtype=torch.float32)
        self.X2 = torch.tensor(X_WAXS, dtype=torch.float32)
        self.X3 = torch.tensor(X_POM, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 4)

    def __len__(self):
        return len(self.X1)

    def __getitem__(self, idx):
        return self.X1[idx], self.X2[idx], self.X3[idx], self.y[idx]

In [186]:
X_IR = (df_IR_norm.values-np.mean(df_IR_norm.values))/np.std(df_IR_norm.values)
y = ((df_pred-df_pred.mean())/df_pred.std()).values
print (X_IR.shape, y.shape)

train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
print (test_groups)

train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
print (test_index)

X_IR_train, y_train=X_IR[train_index].astype("float32"), y[train_index].astype("float32")
X_IR_test, y_test=X_IR[test_index].astype("float32"), y[test_index].astype("float32")
X_IR_train.shape, y_train.shape, X_IR_test.shape, y_test.shape

(153, 2100) (153, 4)
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13]
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13, 94, 91, 97, 63, 75, 82, 68, 83, 54, 81, 64, 145, 142, 148, 114, 126, 133, 119, 134, 105, 132, 115]


((120, 2100), (120, 4), (33, 2100), (33, 4))

In [187]:
X_WAXS = (df_WAXS_norm.values-np.mean(df_WAXS_norm.values))/np.std(df_WAXS_norm.values)
y = ((df_pred-df_pred.mean())/df_pred.std()).values
print (X_WAXS.shape, y.shape)

train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
print (test_groups)

train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
print (test_index)

X_WAXS_train, y_train=X_WAXS[train_index].astype("float32"), y[train_index].astype("float32")
X_WAXS_test, y_test=X_WAXS[test_index].astype("float32"), y[test_index].astype("float32")
X_WAXS_train.shape, y_train.shape, X_WAXS_test.shape, y_test.shape

(153, 1000) (153, 4)
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13]
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13, 94, 91, 97, 63, 75, 82, 68, 83, 54, 81, 64, 145, 142, 148, 114, 126, 133, 119, 134, 105, 132, 115]


((120, 1000), (120, 4), (33, 1000), (33, 4))

In [188]:
X_POM = (POM_list_norm-np.mean(POM_list_norm))/np.std(POM_list_norm)
y = df_pred.values
print (X_POM.shape, y.shape)

train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
print (test_groups)

train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
print (test_index)

X_POM_train, y_train=X_POM[train_index].astype("float32"), y[train_index].astype("float32")
X_POM_test, y_test=X_POM[test_index].astype("float32"), y[test_index].astype("float32")

y_train_mean, y_train_std=np.mean(y_train, axis=0), np.std(y_train, axis=0)
y_train=(y_train-y_train_mean)/y_train_std
y_test=(y_test-y_train_mean)/y_train_std

X_POM_train.shape, y_train.shape, X_POM_test.shape, y_test.shape

(153, 496, 880) (153, 4)
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13]
[43, 40, 46, 12, 24, 31, 17, 32, 3, 30, 13, 94, 91, 97, 63, 75, 82, 68, 83, 54, 81, 64, 145, 142, 148, 114, 126, 133, 119, 134, 105, 132, 115]


((120, 496, 880), (120, 4), (33, 496, 880), (33, 4))

In [189]:
train_All_dataset = AllDataset(X_IR_train, X_WAXS_train, X_POM_train, y_train)
test_All_dataset  = AllDataset(X_IR_test, X_WAXS_test, X_POM_test, y_test)

train_All_loader = DataLoader(train_All_dataset, batch_size=16, shuffle=True)
test_All_loader  = DataLoader(test_All_dataset, batch_size=16, shuffle=False)

# 2. Cross-modal Contrastive Learning

In [210]:
class Encoder1D(nn.Module):
    def __init__(self, input_length, latent):
        super().__init__()

        self.input_length = input_length
        self.latent = latent
        
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 16, kernel_size=1),
            nn.ReLU(),

            nn.Flatten(),

            nn.Linear(16 * (self.input_length // 2), self.latent)
        )

    def forward(self, x):
        return self.net(x)

In [211]:
class Encoder2D(nn.Module):
    def __init__(self, latent):
        super().__init__()

        self.latent = latent
        
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        
            nn.Conv2d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        
            nn.Conv2d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(128, self.latent, kernel_size=1),
            nn.ReLU(),
            
            nn.AdaptiveAvgPool2d((1,1)),
        
            nn.Flatten()
        )
    
    def forward(self, x):
        return self.net(x)

In [ ]:
class NTXentLossMultiModal(torch.nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def ntxent(self, z1, z2):
        """
        z1, z2: (B, D)
        """

        B = z1.size(0)

        z = torch.cat([z1, z2], dim=0)  # (2B, D)

        sim = torch.matmul(z, z.T) / self.temperature

        sim = sim - sim.max(dim=1, keepdim=True).values.detach()

        diag_mask = torch.eye(2 * B, device=z.device, dtype=torch.bool)

        pos_mask = torch.zeros((2 * B, 2 * B),
                               device=z.device,
                               dtype=torch.bool)

        for i in range(B):
            pos_mask[i, B + i] = True
            pos_mask[B + i, i] = True

        log_denom = torch.logsumexp(sim, dim=1)

        pos_sim = sim.masked_fill(~pos_mask, float("-inf"))
        log_pos = torch.logsumexp(pos_sim, dim=1)

        loss = -(log_pos - log_denom).mean()

        return loss

    def forward(self, z_ir, z_wax, z_pom):
    
        B = z_ir.size(0)
    
        loss_ir_wax = self.ntxent(z_ir, z_wax)
        loss_ir_pom = self.ntxent(z_ir, z_pom)
        loss_wax_pom = self.ntxent(z_wax, z_pom)
    
        loss = loss_ir_wax + loss_ir_pom + loss_wax_pom
    
    
        # =========================
        # positive cosine
        # =========================
        sim_ir_wax = torch.matmul(z_ir, z_wax.T)
        sim_ir_pom = torch.matmul(z_ir, z_pom.T)
        sim_wax_pom = torch.matmul(z_wax, z_pom.T)
    
        cos_ir_wax = torch.diag(sim_ir_wax).mean()
        cos_ir_pom = torch.diag(sim_ir_pom).mean()
        cos_wax_pom = torch.diag(sim_wax_pom).mean()
    
    
        # =========================
        # negative cosine
        # =========================
        neg_mask = ~torch.eye(
            B,
            device=z_ir.device,
            dtype=torch.bool
        )
    
        neg_ir_wax = sim_ir_wax[neg_mask].mean()
        neg_ir_pom = sim_ir_pom[neg_mask].mean()
        neg_wax_pom = sim_wax_pom[neg_mask].mean()

        return {
            "loss": loss,
            "loss_ir_wax": loss_ir_wax.detach(),
            "loss_ir_pom": loss_ir_pom.detach(),
            "loss_wax_pom": loss_wax_pom.detach(),
            "cos_ir_wax": cos_ir_wax.detach(),
            "cos_ir_pom": cos_ir_pom.detach(),
            "cos_wax_pom": cos_wax_pom.detach(),
            "neg_ir_wax": neg_ir_wax.detach(),
            "neg_ir_pom": neg_ir_pom.detach(),
            "neg_wax_pom": neg_wax_pom.detach(),
        }

In [ ]:
class MultiModalSimCLR(pl.LightningModule):

    def __init__(self, input_length_IR=512, input_length_WAXS=512, latent=64, proj_dim=32, lr=1e-3, temperature=0.5):
        super().__init__()
        self.save_hyperparameters()

        # =========================
        # encoders
        # =========================
        self.encoder_ir = Encoder1D(input_length_IR, latent)
        self.encoder_wax = Encoder1D(input_length_WAXS, latent)
        self.encoder_pom = Encoder2D(latent)

        # =========================
        # projector
        # Linear -> BN -> ReLU -> Linear -> BN -> ReLU -> Linear
        # =========================

        self.projector_ir = nn.Sequential(
            nn.Linear(latent, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

        self.projector_wax = nn.Sequential(
            nn.Linear(latent, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

        self.projector_pom = nn.Sequential(
            nn.Linear(latent, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

        # loss
        self.criterion = NTXentLossMultiModal(temperature)

        self.train_loss_history = []
        self.train_ir_wax_history = []
        self.train_ir_pom_history = []
        self.train_wax_pom_history = []
        self.cos_ir_wax_history = []
        self.cos_ir_pom_history = []
        self.cos_wax_pom_history = []
        self.neg_ir_wax_history = []
        self.neg_ir_pom_history = []
        self.neg_wax_pom_history = []

        self.val_loss_history = []
        self.val_ir_wax_history = []
        self.val_ir_pom_history = []
        self.val_wax_pom_history = []
        self.val_cos_ir_wax_history = []
        self.val_cos_ir_pom_history = []
        self.val_cos_wax_pom_history = []
        self.val_neg_ir_wax_history = []
        self.val_neg_ir_pom_history = []
        self.val_neg_wax_pom_history = []

    # =========================
    # forward (projection space)
    # =========================
    def forward(self, x_ir, x_wax, x_pom):

        h_ir = self.encoder_ir(x_ir)
        h_wax = self.encoder_wax(x_wax)
        h_pom = self.encoder_pom(x_pom)

        z_ir = self.projector_ir(h_ir)
        z_wax = self.projector_wax(h_wax)
        z_pom = self.projector_pom(h_pom)

        z_ir = F.normalize(z_ir, dim=1)
        z_wax = F.normalize(z_wax, dim=1)
        z_pom = F.normalize(z_pom, dim=1)

        return z_ir, z_wax, z_pom

    def encode1(self, x_ir, x_wax, x_pom):

        h_ir = self.encoder_ir(x_ir)
        h_wax = self.encoder_wax(x_wax)
        h_pom = self.encoder_pom(x_pom)

        return h_ir, h_wax, h_pom

    def encode2(self, x_ir, x_wax, x_pom):

        h_ir = self.encoder_ir(x_ir)
        h_wax = self.encoder_wax(x_wax)
        h_pom = self.encoder_pom(x_pom)

        z_ir = self.projector_ir(h_ir)
        z_wax = self.projector_wax(h_wax)
        z_pom = self.projector_pom(h_pom)

        return z_ir, z_wax, z_pom

    def training_step(self, batch, batch_idx):
        x_ir, x_wax, x_pom, _ = batch    
        z_ir, z_wax, z_pom = self(x_ir, x_wax, x_pom)
        out = self.criterion(z_ir, z_wax, z_pom)
        self.log("train_loss", out["loss"], prog_bar=True, on_step=False, on_epoch=True)
        self.log("loss_ir_wax", out["loss_ir_wax"], on_step=False, on_epoch=True)
        self.log("loss_ir_pom", out["loss_ir_pom"], on_step=False, on_epoch=True)
        self.log("loss_wax_pom", out["loss_wax_pom"], on_step=False, on_epoch=True)

        self.log("cos_ir_wax", out["cos_ir_wax"], on_step=False, on_epoch=True)
        self.log("cos_ir_pom", out["cos_ir_pom"], on_step=False, on_epoch=True)
        self.log("cos_wax_pom", out["cos_wax_pom"], on_step=False, on_epoch=True)
        self.log("neg_ir_wax", out["neg_ir_wax"], on_step=False, on_epoch=True)
        self.log("neg_ir_pom", out["neg_ir_pom"], on_step=False, on_epoch=True)
        self.log("neg_wax_pom", out["neg_wax_pom"], on_step=False, on_epoch=True)

        return out["loss"]

    def validation_step(self, batch, batch_idx):
        x_ir, x_wax, x_pom, _ = batch

        z_ir, z_wax, z_pom = self(x_ir, x_wax, x_pom)
        out = self.criterion(z_ir, z_wax, z_pom)
        self.log("val_loss", out["loss"], prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_loss_ir_wax", out["loss_ir_wax"], on_step=False, on_epoch=True)
        self.log("val_loss_ir_pom", out["loss_ir_pom"], on_step=False, on_epoch=True)
        self.log("val_loss_wax_pom", out["loss_wax_pom"], on_step=False, on_epoch=True)

        self.log("val_cos_ir_wax", out["cos_ir_wax"], on_step=False, on_epoch=True)
        self.log("val_cos_ir_pom", out["cos_ir_pom"], on_step=False, on_epoch=True)
        self.log("val_cos_wax_pom", out["cos_wax_pom"], on_step=False, on_epoch=True)
        self.log("val_neg_ir_wax", out["neg_ir_wax"], on_step=False, on_epoch=True)
        self.log("val_neg_ir_pom", out["neg_ir_pom"], on_step=False, on_epoch=True)
        self.log("val_neg_wax_pom", out["neg_wax_pom"], on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

    def on_train_epoch_end(self):
        if self.current_epoch%10==0:
            print(f"Epoch {self.current_epoch + 1}/{self.trainer.max_epochs}")
        metrics = self.trainer.callback_metrics
        if "train_loss" in metrics:
            self.train_loss_history.append(float(metrics["train_loss"].cpu()))

        if "loss_ir_wax" in metrics:
            self.train_ir_wax_history.append(float(metrics["loss_ir_wax"].cpu()))

        if "loss_ir_pom" in metrics:
            self.train_ir_pom_history.append(float(metrics["loss_ir_pom"].cpu()))

        if "loss_wax_pom" in metrics:
            self.train_wax_pom_history.append(float(metrics["loss_wax_pom"].cpu()))

        if "cos_ir_wax" in metrics:
            self.cos_ir_wax_history.append(float(metrics["cos_ir_wax"].cpu()))

        if "cos_ir_pom" in metrics:
            self.cos_ir_pom_history.append(float(metrics["cos_ir_pom"].cpu()))

        if "cos_wax_pom" in metrics:
            self.cos_wax_pom_history.append(float(metrics["cos_wax_pom"].cpu()))

        if "neg_ir_wax" in metrics:
            self.neg_ir_wax_history.append(float(metrics["neg_ir_wax"].cpu()))

        if "neg_ir_pom" in metrics:
            self.neg_ir_pom_history.append(float(metrics["neg_ir_pom"].cpu()))

        if "neg_wax_pom" in metrics:
            self.neg_wax_pom_history.append(float(metrics["neg_wax_pom"].cpu()))

    def on_validation_epoch_end(self):

        metrics = self.trainer.callback_metrics

        if "val_loss" in metrics:
            self.val_loss_history.append(float(metrics["val_loss"].cpu()))

        if "val_loss_ir_wax" in metrics:
            self.val_ir_wax_history.append(float(metrics["val_loss_ir_wax"].cpu()))

        if "val_loss_ir_pom" in metrics:
            self.val_ir_pom_history.append(float(metrics["val_loss_ir_pom"].cpu()))

        if "val_loss_wax_pom" in metrics:
            self.val_wax_pom_history.append(float(metrics["val_loss_wax_pom"].cpu()))

        if "val_cos_ir_wax" in metrics:
            self.val_cos_ir_wax_history.append(float(metrics["val_cos_ir_wax"].cpu()))

        if "val_cos_ir_pom" in metrics:
            self.val_cos_ir_pom_history.append(float(metrics["val_cos_ir_pom"].cpu()))

        if "val_cos_wax_pom" in metrics:
            self.val_cos_wax_pom_history.append(float(metrics["val_cos_wax_pom"].cpu()))

        if "val_neg_ir_wax" in metrics:
            self.val_neg_ir_wax_history.append(float(metrics["val_neg_ir_wax"].cpu()))

        if "val_neg_ir_pom" in metrics:
            self.val_neg_ir_pom_history.append(float(metrics["val_neg_ir_pom"].cpu()))

        if "val_neg_wax_pom" in metrics:
            self.val_neg_wax_pom_history.append(float(metrics["val_neg_wax_pom"].cpu()))

In [ ]:
input_length_IR=X_IR_train.shape[1]
input_length_WAXS=X_WAXS_train.shape[1]

for latent in latent_list:
    train_All_dataset = AllDataset(X_IR_train, X_WAXS_train, X_POM_train, y_train)
    test_All_dataset  = AllDataset(X_IR_test, X_WAXS_test, X_POM_test, y_test)
    
    train_All_loader = DataLoader(train_All_dataset, batch_size=16, shuffle=True, drop_last=True)
    test_All_loader  = DataLoader(test_All_dataset, batch_size=16, shuffle=False, drop_last=True)
    
    model_multi = MultiModalSimCLR(input_length_IR=input_length_IR, input_length_WAXS=input_length_WAXS,
                                   latent=latent, proj_dim=proj_dim, lr=1e-4, temperature=0.2)
    
    trainer_multi = pl.Trainer(
        max_epochs=50,
        enable_progress_bar=True,
        accelerator="auto",
        logger=True
    )
    
    trainer_multi.fit(model_multi, train_All_loader, test_All_loader)
    
    ckpt = {
            "model_state": model_multi.state_dict(),
            "train_loss_history": model_multi.train_loss_history,
            "train_ir_wax_history": model_multi.train_ir_wax_history,
            "train_ir_pom_history": model_multi.train_ir_pom_history,
            "train_wax_pom_history": model_multi.train_wax_pom_history,
            "cos_ir_wax_history": model_multi.cos_ir_wax_history,
            "cos_ir_pom_history": model_multi.cos_ir_pom_history,
            "cos_wax_pom_history": model_multi.cos_wax_pom_history,
            "neg_ir_wax_history": model_multi.neg_ir_wax_history,
            "neg_ir_pom_history": model_multi.neg_ir_pom_history,
            "neg_wax_pom_history": model_multi.neg_wax_pom_history,
            "val_loss_history": model_multi.val_loss_history,
            "val_ir_wax_history": model_multi.val_ir_wax_history,
            "val_ir_pom_history": model_multi.val_ir_pom_history,
            "val_wax_pom_history": model_multi.val_wax_pom_history,
            "val_cos_ir_wax_history": model_multi.val_cos_ir_wax_history,
            "val_cos_ir_pom_history": model_multi.val_cos_ir_pom_history,
            "val_cos_wax_pom_history": model_multi.val_cos_wax_pom_history,
            "val_neg_ir_wax_history": model_multi.val_neg_ir_wax_history,
            "val_neg_ir_pom_history": model_multi.val_neg_ir_pom_history,
            "val_neg_wax_pom_history": model_multi.val_neg_wax_pom_history,
            "hparams": model_multi.hparams
        }
    torch.save(ckpt, path + "/model/Analysis2/"+model+"/Multi_"+model+"_"+str(latent)+"_"+str(seed)+".pth")

# 3. Latent Embedding Extraction

In [98]:
class AllDataset(Dataset):
    def __init__(self, X_IR, X_WAXS, X_POM, y):
        # (N, length) → (N, 1, length)
        X_IR = X_IR[:, None, :]
        X_WAXS = X_WAXS[:, None, :]
        X_POM = X_POM[:, None, :]
        
        self.X1 = torch.tensor(X_IR, dtype=torch.float32)
        self.X2 = torch.tensor(X_WAXS, dtype=torch.float32)
        self.X3 = torch.tensor(X_POM, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 4)

    def __len__(self):
        return len(self.X1)

    def __getitem__(self, idx):
        return self.X1[idx], self.X2[idx], self.X3[idx], self.y[idx]

In [62]:
for latent in latent_list:
    model_CL2=MultiModalSimCLR(input_length_IR=input_length_IR, input_length_WAXS=input_length_WAXS,
                               latent=latent, proj_dim=proj_dim)
    model_CL2.load_state_dict(torch.load(path + "/model/Analysis2/"+model+"/Multi_"+model+"_"+str(latent)+"_"+str(seed)+".pth", map_location="cpu",
                                         weights_only=False)["model_state"])
    model_CL2.eval()

    train_All_dataset = AllDataset(X_IR_train, X_WAXS_train, X_POM_train, y_train)
    test_All_dataset  = AllDataset(X_IR_test, X_WAXS_test, X_POM_test, y_test)
    
    train_All_loader = DataLoader(train_All_dataset, batch_size=1, shuffle=False)
    test_All_loader  = DataLoader(test_All_dataset, batch_size=1, shuffle=False)

    mid1_IR_train, mid1_WAXS_train, mid1_POM_train, y_train_real = [], [], [], [] 
    mid1_IR_test,  mid1_WAXS_test,  mid1_POM_test, y_test_real  = [], [], [], [] 
    mid2_IR_train, mid2_WAXS_train, mid2_POM_train, y_train_real = [], [], [], [] 
    mid2_IR_test,  mid2_WAXS_test,  mid2_POM_test, y_test_real  = [], [], [], [] 
    
    with torch.no_grad():
        for x_IR, x_WAXS, x_POM, y in train_All_loader:
            mid_IR, mid_WAXS, mid_POM=model_CL2.encode1(x_IR, x_WAXS, x_POM)
            mid1_IR_train.append(mid_IR.cpu().numpy()[0])
            mid1_WAXS_train.append(mid_WAXS.cpu().numpy()[0])
            mid1_POM_train.append(mid_POM.cpu().numpy()[0])
            mid_IR, mid_WAXS, mid_POM=model_CL2.encode2(x_IR, x_WAXS, x_POM)
            mid2_IR_train.append(mid_IR.cpu().numpy()[0])
            mid2_WAXS_train.append(mid_WAXS.cpu().numpy()[0])
            mid2_POM_train.append(mid_POM.cpu().numpy()[0])
            y_train_real.append(y.cpu().numpy()[0])
    
        for x_IR, x_WAXS, x_POM, y in test_All_loader:
            mid_IR, mid_WAXS, mid_POM=model_CL2.encode1(x_IR, x_WAXS, x_POM)
            mid1_IR_test.append(mid_IR.cpu().numpy()[0])
            mid1_WAXS_test.append(mid_WAXS.cpu().numpy()[0])
            mid1_POM_test.append(mid_POM.cpu().numpy()[0])
            mid_IR, mid_WAXS, mid_POM=model_CL2.encode2(x_IR, x_WAXS, x_POM)
            mid2_IR_test.append(mid_IR.cpu().numpy()[0])
            mid2_WAXS_test.append(mid_WAXS.cpu().numpy()[0])
            mid2_POM_test.append(mid_POM.cpu().numpy()[0])
            y_test_real.append(y.cpu().numpy()[0])
    
    X_mid1_train = pd.concat([pd.DataFrame(mid1_IR_train), pd.DataFrame(mid1_WAXS_train), pd.DataFrame(mid1_POM_train)], axis=1)
    X_mid1_test  = pd.concat([pd.DataFrame(mid1_IR_test),  pd.DataFrame(mid1_WAXS_test),  pd.DataFrame(mid1_POM_test)],  axis=1)
    X_mid2_train = pd.concat([pd.DataFrame(mid2_IR_train), pd.DataFrame(mid2_WAXS_train), pd.DataFrame(mid2_POM_train)], axis=1)
    X_mid2_test  = pd.concat([pd.DataFrame(mid2_IR_test),  pd.DataFrame(mid2_WAXS_test),  pd.DataFrame(mid2_POM_test)],  axis=1)

    y_train_real=np.array(y_train_real)
    y_test_real =np.array(y_test_real)

    X_mid1_train.to_csv(path+'/data/mid/'+model+'/mid_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')
    X_mid1_test.to_csv(path+'/data/mid/'+model+'/mid_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')
    X_mid2_train.to_csv(path+'/data/mid/'+model+'/mid_train_'+model+'_'+str(latent)+"_"+str(seed)+'-2.csv')
    X_mid2_test.to_csv(path+'/data/mid/'+model+'/mid_test_'+model+'_'+str(latent)+"_"+str(seed)+'-2.csv')
    pd.DataFrame(y_train_real).to_csv(path+'/data/mid/'+model+'/y_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')
    pd.DataFrame(y_test_real).to_csv(path+'/data/mid/'+model+'/y_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv')

# 4. LASSO

In [161]:
y_name=["Deg","Strain", "Young", "CrysDeg"]
indicator_name=["Deg,train", "Deg,test","Strain,train", "Strain,test","Young,train", "Young,test","CryDeg,train", "CryDeg,test"]

In [162]:
df_result_sum=pd.DataFrame(columns=indicator_name)
df_result_sum2=pd.DataFrame(columns=indicator_name)

,"Deg,train","Deg,test","Strain,train","Strain,test","Young,train","Young,test","CryDeg,train","CryDeg,test"


In [163]:
def reg(kind="all", method="Lasso", df_result_sum=df_result_sum, hosei=True):
    for latent in latent_list:
        result=[]
        X_mid_train=pd.read_csv(path+'/data/mid/'+model+'/mid_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        X_mid_test =pd.read_csv(path+'/data/mid/'+model+'/mid_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        if hosei:
            eps = np.finfo(np.float64).eps
            ave=copy.copy(np.repeat([X_mid_train.iloc[:,:latent].mean().mean()+eps,X_mid_train.iloc[:,latent:latent*2].mean().mean()+eps,X_mid_train.iloc[:,latent*2:].mean().mean()+eps], latent))
            X_mid_train=X_mid_train/ave
            X_mid_test=X_mid_test/ave
        y_train_real=pd.read_csv(path+'/data/mid/'+model+'/y_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        y_test_real =pd.read_csv(path+'/data/mid/'+model+'/y_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        if kind=="all":
            X_mid_train, X_mid_test=X_mid_train, X_mid_test
        elif kind=="IR":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,:latent], X_mid_test.iloc[:,:latent]
        elif kind=="WAXS":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,latent:latent*2], X_mid_test.iloc[:,latent:latent*2]
        elif kind=="POM":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,latent*2:], X_mid_test.iloc[:,latent*2:]
        print (X_mid_train.shape, X_mid_test.shape, y_train_real.shape, y_test_real.shape)
            
        non_zero_coef=[]
        for i in range(4):
            import warnings
            from sklearn.exceptions import ConvergenceWarning
            warnings.filterwarnings("ignore", category=ConvergenceWarning)
            lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
            lasso_cv.fit(X_mid_train, y_train_real[str(i)])
            lasso = Lasso(alpha=lasso_cv.alpha_)
            lasso.fit(X_mid_train, y_train_real[str(i)])
            y_train_pred=lasso.predict(X_mid_train)
            y_test_pred=lasso.predict(X_mid_test)
            non_zero_coef.append(int(np.sum(lasso.coef_ != 0)))
            if i==3:
                print (non_zero_coef)
        
            result.append(r2_score(y_train_real[str(i)], y_train_pred))
            result.append(r2_score(y_test_real[str(i)], y_test_pred))
        df_result=pd.DataFrame(result,index=indicator_name, columns=[kind+','+method+','+str(latent)])
        df_result_sum=pd.concat([df_result_sum, df_result.T])
        
    return df_result_sum

In [ ]:
def reg2(kind="all", method="Lasso", df_result_sum=df_result_sum2, proj_dim=proj_dim, hosei=True):
    for latent in latent_list:
        result=[]
        X_mid_train=pd.read_csv(path+'/data/mid/'+model+'/mid_train_'+model+'_'+str(latent)+"_"+str(seed)+'-2.csv').iloc[:,1:]
        X_mid_test =pd.read_csv(path+'/data/mid/'+model+'/mid_test_'+model+'_'+str(latent)+"_"+str(seed)+'-2.csv').iloc[:,1:]
        if hosei:
            eps = np.finfo(np.float64).eps
            ave=copy.copy(np.repeat([X_mid_train.iloc[:,:proj_dim].mean().mean()+eps,X_mid_train.iloc[:,proj_dim:proj_dim*2].mean().mean()+eps,X_mid_train.iloc[:,proj_dim*2:].mean().mean()+eps], proj_dim))
            X_mid_train=X_mid_train/ave
            X_mid_test=X_mid_test/ave
        y_train_real=pd.read_csv(path+'/data/mid/'+model+'/y_train_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        y_test_real =pd.read_csv(path+'/data/mid/'+model+'/y_test_'+model+'_'+str(latent)+"_"+str(seed)+'.csv').iloc[:,1:]
        if kind=="all":
            X_mid_train, X_mid_test=X_mid_train, X_mid_test
        elif kind=="IR":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,:proj_dim], X_mid_test.iloc[:,:proj_dim]
        elif kind=="WAXS":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,proj_dim:proj_dim*2], X_mid_test.iloc[:,proj_dim:proj_dim*2]
        elif kind=="POM":
            X_mid_train, X_mid_test=X_mid_train.iloc[:,proj_dim*2:], X_mid_test.iloc[:,proj_dim*2:]
        print (X_mid_train.shape, X_mid_test.shape, y_train_real.shape, y_test_real.shape)
            
        non_zero_coef=[]
        for i in range(4):
            import warnings
            from sklearn.exceptions import ConvergenceWarning
            warnings.filterwarnings("ignore", category=ConvergenceWarning)
            lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
            lasso_cv.fit(X_mid_train, y_train_real[str(i)])
            lasso = Lasso(alpha=lasso_cv.alpha_)
            lasso.fit(X_mid_train, y_train_real[str(i)])
            y_train_pred=lasso.predict(X_mid_train)
            y_test_pred=lasso.predict(X_mid_test)
            non_zero_coef.append(int(np.sum(lasso.coef_ != 0)))
            if i==3:
                print (non_zero_coef)
              
            result.append(r2_score(y_train_real[str(i)], y_train_pred))
            result.append(r2_score(y_test_real[str(i)], y_test_pred))
        df_result=pd.DataFrame(result,index=indicator_name, columns=[kind+','+method+','+str(latent)])
        df_result_sum=pd.concat([df_result_sum, df_result.T])
        
    return df_result_sum

In [ ]:
df_result_sum=reg(kind="all", method="Lasso", df_result_sum=df_result_sum)
df_result_sum2=reg2(kind="all", method="Lasso", df_result_sum=df_result_sum2)

In [ ]:
df_result_sum=reg(kind="IR", method="Lasso", df_result_sum=df_result_sum)
df_result_sum2=reg2(kind="IR", method="Lasso", df_result_sum=df_result_sum2)

In [ ]:
df_result_sum=reg(kind="WAXS", method="Lasso", df_result_sum=df_result_sum)
df_result_sum2=reg2(kind="WAXS", method="Lasso", df_result_sum=df_result_sum2)

In [ ]:
df_result_sum=reg(kind="POM", method="Lasso", df_result_sum=df_result_sum)
df_result_sum2=reg2(kind="POM", method="Lasso", df_result_sum=df_result_sum2)

In [185]:
df_result_sum.to_csv(path+'/regression/R2_'+model+'_'+str(seed)+'.csv')
df_result_sum2.to_csv(path+'/regression/R2_'+model+'_'+str(seed)+'-2.csv')